# Connect to k.LAB
Load a deployment profile, connect, and see which services are ready.
This notebook uses a local TEST authority and real local services, not a
production Hub. `notebook_setup` loads credentials privately; it does not fake
any service responses. No credential value is displayed or saved in this file.

In [1]:
import pandas as pd
from IPython.display import display
from klab_client import connect, DeploymentProfile
from notebook_setup import profile, certificate, issued_token

deployment = profile()
display(pd.DataFrame([{"service": s.service_type, "URL": s.url}
                      for s in deployment.services]))

,service,URL
0,reasoner,http://127.0.0.1:8091/reasoner
1,resources,http://127.0.0.1:8092/resources
2,resolver,http://127.0.0.1:8093/resolver
3,runtime,http://127.0.0.1:8094/runtime


## Connect with an ENGINE certificate
The provider exchanges the certificate for service credentials. The connection
checks the configured services and initializes the client for this caller.

In [2]:
connection = connect(deployment, credentials=certificate())
connection.wait_ready(timeout=30)
display(pd.DataFrame(connection.catalog.summary())[
    ["type", "trusted", "reachable", "available", "operational"]])
print("Signed in as:", connection.identity.principal)

,type,trusted,reachable,available,operational
0,reasoner,True,True,True,True
1,resources,True,True,True,True
2,resolver,True,True,True,True
3,runtime,True,True,True,True


Signed in as: python-scientist


## See what is available
The catalog and readiness explanation come from the connected deployment.

In [3]:
display(connection.explain_readiness())
models = connection.client.resources.list("MODEL")
display(models[:8])
assert models

'Connection dependencies not ready: '

[KlabAsset(urn='python.scenario.elevation-quantifier'),
 KlabAsset(urn='python.lexical.slope-quantifier'),
 KlabAsset(urn='python.lexical.bathymetric_depth-quantifier'),
 KlabAsset(urn='python.fixture.elevation-quantifier'),
 KlabAsset(urn='python.fixture.aspect-quantifier'),
 KlabAsset(urn='python.fixture.bathymetric_depth-quantifier')]

## Keep your profile, not your credentials
Save the deployment choices so another Python process can load them.

In [4]:
deployment.save("deployment.json")
reloaded = DeploymentProfile.load("deployment.json")
assert reloaded.to_dict() == deployment.to_dict()
print("Saved profile:", reloaded.name)
print("Required services:", ", ".join(reloaded.required_services))
connection.refresh()
connection.renew()
assert all(row["operational"] for row in connection.catalog.summary())
connection.close()

Saved profile: owned-controlled-TEST
Required services: runtime, reasoner, resources, resolver


## Already have issued credentials?
Use the token provider instead. Secrets still come from private configuration,
not the notebook. This is a new connection to the same deployment.

In [5]:
with connect(reloaded, credentials=issued_token()) as token_connection:
    token_connection.wait_ready(timeout=30)
    display(pd.DataFrame(token_connection.catalog.summary())[
        ["type", "available", "operational"]])
    assert token_connection.identity.principal == "python-scientist"
print("Both connections closed locally; no remote work was released.")

,type,available,operational
0,reasoner,True,True
1,resources,True,True
2,resolver,True,True
3,runtime,True,True


Both connections closed locally; no remote work was released.
